# 🔎 TruthLens AI — Transformer-Based Fake News Detection

This notebook demonstrates an end-to-end NLP classification workflow using **DistilBERT**, **Hugging Face Transformers**, **PyTorch**, and **Scikit-learn**.

**Pipeline:** Dataset → EDA → Train/Validation Split → Tokenization → DistilBERT Fine-tuning → Evaluation → Custom Prediction → Save Model

> **Important:** The small CSV included with the project is only for testing the pipeline. Use a large, properly curated dataset for meaningful model performance.


## 1. Install required libraries


In [ ]:
# Run this once if the packages are not installed.
# In Jupyter:
# %pip install torch transformers datasets accelerate pandas scikit-learn matplotlib


## 2. Import libraries


In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 3. Load the dataset


In [ ]:
DATA_PATH = "data/news.csv"

df = pd.read_csv(DATA_PATH)
df.head()


In [ ]:
print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum())
print("\nLabel counts:")
print(df["label"].value_counts())


## 4. Basic EDA

Labels used in this project:

- `0` = REAL
- `1` = FAKE


In [ ]:
label_counts = df["label"].value_counts().sort_index()

plt.figure(figsize=(6, 4))
plt.bar(["REAL (0)", "FAKE (1)"], [label_counts.get(0, 0), label_counts.get(1, 0)])
plt.title("Class Distribution")
plt.ylabel("Number of Articles")
plt.show()


In [ ]:
df["text_length"] = df["text"].astype(str).str.len()

print(df["text_length"].describe())

plt.figure(figsize=(7, 4))
plt.hist(df["text_length"], bins=20)
plt.title("News Text Length Distribution")
plt.xlabel("Characters")
plt.ylabel("Frequency")
plt.show()


## 5. Prepare and split the data


In [ ]:
df = df[["text", "label"]].dropna().copy()
df["text"] = df["text"].astype(str)
df["label"] = df["label"].astype(int)

assert set(df["label"].unique()).issubset({0, 1}), "Labels must be 0 or 1."

train_df, val_df = train_test_split(
    df,
    test_size=0.25,
    random_state=42,
    stratify=df["label"],
)

print("Training examples:", len(train_df))
print("Validation examples:", len(val_df))


## 6. Load DistilBERT tokenizer


In [ ]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

example = tokenizer(
    "This is an example news article.",
    truncation=True,
    padding="max_length",
    max_length=64,
)

print("Input IDs:", example["input_ids"][:15])
print("Attention mask:", example["attention_mask"][:15])


## 7. Convert to Hugging Face datasets and tokenize


In [ ]:
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256,
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

print(train_dataset)


## 8. Load the Transformer classification model


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
)

print(model.__class__.__name__)


## 9. Define evaluation metrics


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        zero_division=0,
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


## 10. Fine-tune DistilBERT

For the tiny demonstration CSV this only proves that the pipeline works. Replace the dataset with thousands of representative examples before interpreting the metrics.


In [ ]:
training_args = TrainingArguments(
    output_dir="training_output",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()


## 11. Evaluate the model


In [ ]:
evaluation_results = trainer.evaluate()

for key, value in evaluation_results.items():
    print(f"{key}: {value}")


In [ ]:
prediction_output = trainer.predict(val_dataset)
predicted_labels = np.argmax(prediction_output.predictions, axis=-1)
true_labels = prediction_output.label_ids

print(
    classification_report(
        true_labels,
        predicted_labels,
        target_names=["REAL", "FAKE"],
        zero_division=0,
    )
)


## 12. Confusion matrix


In [ ]:
cm = confusion_matrix(true_labels, predicted_labels, labels=[0, 1])

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["REAL", "FAKE"],
)

disp.plot()
plt.title("DistilBERT Confusion Matrix")
plt.show()


## 13. Predict custom news


In [ ]:
def predict_news(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256,
    )

    device = next(model.parameters()).device
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=1)[0].cpu()
    prediction = torch.argmax(probabilities).item()

    return {
        "prediction": "FAKE" if prediction == 1 else "REAL",
        "confidence": float(probabilities[prediction]),
        "real_probability": float(probabilities[0]),
        "fake_probability": float(probabilities[1]),
    }

sample_news = "Government announces a new education program after cabinet approval."
result = predict_news(sample_news)

print("News:", sample_news)
print("Prediction:", result["prediction"])
print(f"Confidence: {result['confidence'] * 100:.2f}%")
print(f"REAL probability: {result['real_probability'] * 100:.2f}%")
print(f"FAKE probability: {result['fake_probability'] * 100:.2f}%")


## 14. Save the trained model


In [ ]:
MODEL_OUTPUT = "model"

os.makedirs(MODEL_OUTPUT, exist_ok=True)

model.save_pretrained(MODEL_OUTPUT)
tokenizer.save_pretrained(MODEL_OUTPUT)

print(f"Model and tokenizer saved to: {MODEL_OUTPUT}")


## 15. Next step: Run the Streamlit application

After saving the model, open a terminal in the project folder and run:

```powershell
streamlit run app.py
```

### Portfolio upgrades

For a stronger project, train on a large dataset, compare **DistilBERT vs BERT vs RoBERTa**, add ROC-AUC, use a held-out test set, and add evidence retrieval so the application does not rely only on learned writing patterns.
